# Production Drug-Intelligence Agent: LangGraph + ChEMBL + OpenAI
**Author:** Himanshu Goel | [Website](https://hgoelgithub.github.io)

This notebook starts with the small prototype for comparison, then runs a production-oriented
implementation from `chembl_agent.py`. The production path adds validated data contracts,
bounded retries and timeouts, concurrent retrieval, caching, structured logs, trace IDs,
OpenAI Responses API structured output, deterministic fallback behavior, and offline tests.

> Research-use decision support only. This workflow is not medical, clinical, or regulatory advice.

In [ ]:
%pip install "openai>=3.6" "langgraph>=1.0" "pydantic>=2.7" "requests>=2.32" python-dotenv -q

In [ ]:
# Load OPENAI_API_KEY from a .env file. The .env can live next to this notebook
# OR in any parent folder (e.g. the repo root) — we walk upward until we find one.
# .env line format (no quotes, no spaces):   OPENAI_API_KEY=sk-...
import os
from pathlib import Path

def load_env_upwards() -> str | None:
    try:
        from dotenv import load_dotenv
    except ModuleNotFoundError:
        print("python-dotenv not installed — run the install cell above")
        return None
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / ".env"
        if cand.is_file():
            load_dotenv(cand, override=False)
            return str(cand)
    return None

found = load_env_upwards()
key = os.environ.get("OPENAI_API_KEY", "")
print("Environment file loaded:", bool(found))
print("OPENAI_API_KEY configured:", bool(key))  # Never print any part of a credential.

In [ ]:
import os
import json
import statistics
import operator
from typing import TypedDict, Annotated

import requests

PUBCHEM = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
CHEMBL  = "https://www.ebi.ac.uk/chembl/api/data"
HERG_TARGET = "CHEMBL240"   # KCNH2 / hERG potassium channel

# ── PubChem: name → structure ────────────────────────────────
def get_smiles(compound_name: str) -> dict:
    url = (f"{PUBCHEM}/compound/name/{compound_name}"
           f"/property/SMILES,MolecularWeight,MolecularFormula/JSON")
    r = requests.get(url, timeout=15)
    if not r.ok:
        return {"error": f"PubChem: not found ({compound_name})"}
    p = r.json()["PropertyTable"]["Properties"][0]
    return {
        "name":    compound_name,
        "cid":     p.get("CID"),
        # PubChem renamed the property key to "SMILES"; keep a fallback for older mirrors
        "smiles":  p.get("SMILES") or p.get("ConnectivitySMILES") or p.get("CanonicalSMILES", ""),
        "mw":      float(p["MolecularWeight"]) if p.get("MolecularWeight") else None,
        "formula": p.get("MolecularFormula", ""),
    }

# ── ChEMBL: name → molecule id ─────────────────────────────
def get_chembl_id(compound_name: str) -> dict:
    r = requests.get(f"{CHEMBL}/molecule/search.json",
                     params={"q": compound_name, "limit": 1}, timeout=20)
    if not r.ok or not r.json().get("molecules"):
        return {"error": f"ChEMBL: no match ({compound_name})"}
    m = r.json()["molecules"][0]
    parent = (m.get("molecule_hierarchy") or {}).get("parent_chembl_id")
    props  = m.get("molecule_properties") or {}
    return {
        "chembl_id":  parent or m["molecule_chembl_id"],   # salt-stripped parent for activity lookups
        "pref_name":  m.get("pref_name"),
        "max_phase":  m.get("max_phase"),
        "alogp":      props.get("alogp"),
    }

# ── ChEMBL: molecule id → measured activities ────────────────
def get_chembl_activities(chembl_id: str, target_id: str = None, limit: int = 25) -> list:
    params = {
        "molecule_chembl_id":    chembl_id,
        "pchembl_value__isnull": "false",   # only quantitative, cross-comparable values
        "limit":                 limit,
    }
    if target_id:
        params["target_chembl_id"] = target_id
    r = requests.get(f"{CHEMBL}/activity.json", params=params, timeout=20)
    if not r.ok:
        return []
    out = []
    for a in r.json().get("activities", []):
        try:
            val = float(a["standard_value"])
        except (TypeError, ValueError, KeyError):
            continue
        out.append({
            "type":    a.get("standard_type"),
            "value":   val,
            "units":   a.get("standard_units"),
            "pchembl": float(a["pchembl_value"]) if a.get("pchembl_value") else None,
            "target":  a.get("target_pref_name", ""),
            "assay":   (a.get("assay_description") or "")[:90],
        })
    return out

def summarize_activities(acts: list) -> dict:
    """Collapse a list of activities into headline numbers."""
    if not acts:
        return {"n": 0}
    pchembls = [a["pchembl"] for a in acts if a["pchembl"] is not None]
    by_type = {}
    for a in acts:
        by_type.setdefault(a["type"], []).append(a["value"])
    return {
        "n":              len(acts),
        "n_targets":      len({a["target"] for a in acts}),
        "median_pchembl": round(statistics.median(pchembls), 2) if pchembls else None,
        "potency_nM_by_type": {t: round(statistics.median(v), 1) for t, v in by_type.items()},
    }

# ── smoke test: real numbers straight from the API ──────────
demo = get_chembl_id("terfenadine")
print("ChEMBL lookup:", demo)
herg = get_chembl_activities(demo["chembl_id"], target_id=HERG_TARGET, limit=10)
print(f"\nTerfenadine hERG activities: {len(herg)} rows")
for a in herg[:5]:
    print(f"  {a['type']:>5} = {a['value']:>10,.1f} {a['units']:<4} "
          f"pChEMBL={a['pchembl']}  ({a['target']})")
print("\nHeadline stats:", json.dumps(summarize_activities(herg), indent=2))

## Production implementation

The reusable implementation lives in `chembl_agent.py`; the notebook is only the operator-facing walkthrough.
It keeps deterministic retrieval separate from probabilistic synthesis and validates every boundary with Pydantic.
OpenAI credentials come from `OPENAI_API_KEY`; never paste or commit a key into this notebook. The default model
can be changed with `OPENAI_MODEL`. With no key—or if OpenAI is temporarily unavailable—the workflow returns a
conservative deterministic assessment rather than failing or inventing evidence.

In [ ]:
from pprint import pprint
from chembl_agent import DrugIntelligenceService, Settings, configure_logging

configure_logging()
settings = Settings(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    request_timeout_seconds=20.0,
    max_activities=100,
    openai_timeout_seconds=60.0,
    openai_max_retries=3,
    store_openai_response=False,
)
service = DrugIntelligenceService(settings)
print("Production graph ready; OpenAI enabled:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
result = service.assess("terfenadine")

print("Request ID:", result["request_id"])
print("Assessment:")
pprint(result["assessment"])
print("\nEvidence summary:")
pprint({
    "chembl": result.get("chembl"),
    "activity_summary": result.get("evidence", {}).get("activity_summary"),
    "herg_summary": result.get("evidence", {}).get("herg_summary"),
})
print("\nAudit trail:")
pprint(result.get("audit", []))
if result.get("errors"):
    print("\nNon-fatal warnings:")
    pprint(result["errors"])

## Deployment checklist

- Inject `OPENAI_API_KEY` through a secrets manager; do not use `.env` in deployed workloads.
- Pin and scan dependencies, package `chembl_agent.py`, and run the offline contract tests in CI.
- Export structured logs and request IDs to your observability platform; add latency/error SLOs.
- Add a persistent cache with explicit TTLs and upstream-data provenance for multi-instance deployments.
- Require human review for decisions with safety or regulatory consequences.
- Add golden-set and adversarial evaluations before changing prompts, models, or retrieval logic.

Run the included offline checks with: `python -m unittest test_chembl_agent.py` (or use your CI test runner).

## Prototype baseline (for comparison)

The graph runs four nodes in sequence. Node 4 calls OpenAI through
`langchain_openai.ChatOpenAI`, so set an `OPENAI_API_KEY` in your environment
first:

```python
import os; os.environ["OPENAI_API_KEY"] = "sk-..."
```

If the key is missing the node degrades gracefully and returns the compiled
facts instead of an LLM narrative, so the graph still runs end to end.

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage


class AgentState(TypedDict):
    compound:    str
    structure:   dict
    chembl:      dict
    bioactivity: list
    herg:        list
    facts:       str
    summary:     str
    messages:    Annotated[list, operator.add]


def node_fetch_structure(state: AgentState) -> dict:
    print(f"[1] PubChem structure for {state['compound']}...")
    s = get_smiles(state["compound"])
    return {"structure": s,
            "messages": [f"structure: {s.get('formula', '?')} MW={s.get('mw', '?')}"]}


def node_fetch_bioactivity(state: AgentState) -> dict:
    print(f"[2] ChEMBL bioactivity for {state['compound']}...")
    cid = get_chembl_id(state["compound"])
    acts, herg = [], []
    if "chembl_id" in cid:
        acts = get_chembl_activities(cid["chembl_id"], limit=25)
        herg = get_chembl_activities(cid["chembl_id"], target_id=HERG_TARGET, limit=25)
    return {
        "chembl": cid, "bioactivity": acts, "herg": herg,
        "messages": [f"chembl {cid.get('chembl_id', '?')}: "
                     f"{len(acts)} activities, {len(herg)} hERG"],
    }


def node_compile_facts(state: AgentState) -> dict:
    print("[3] Compiling measured facts...")
    s, c = state["structure"], state.get("chembl", {})
    alla = summarize_activities(state.get("bioactivity", []))
    herg = summarize_activities(state.get("herg", []))
    lines = [
        f"Compound: {state['compound']}  (ChEMBL {c.get('chembl_id', 'N/A')}, "
        f"max_phase={c.get('max_phase', 'N/A')})",
        f"Formula {s.get('formula', 'N/A')}, MW {s.get('mw', 'N/A')} Da, "
        f"cLogP {c.get('alogp', 'N/A')}",
        f"SMILES {s.get('smiles', 'N/A')}",
        f"ChEMBL bioactivity: {alla['n']} measured values across "
        f"{alla.get('n_targets', 0)} targets; median pChEMBL "
        f"{alla.get('median_pchembl', 'N/A')}",
        f"  median potency (nM) by assay type: {alla.get('potency_nM_by_type', {})}",
    ]
    if herg["n"]:
        lines.append(
            f"hERG (KCNH2) liability: {herg['n']} measurements, median pChEMBL "
            f"{herg['median_pchembl']}, median IC50 {herg.get('potency_nM_by_type', {})} nM")
    else:
        lines.append("hERG (KCNH2) liability: no quantitative ChEMBL data")
    facts = "\n".join(lines)
    print(facts)
    return {"facts": facts, "messages": ["facts compiled"]}


def node_llm_summary(state: AgentState) -> dict:
    print("[4] Asking the LLM for a risk summary...")
    if not os.environ.get("OPENAI_API_KEY"):
        return {"summary": "[no OPENAI_API_KEY set — returning raw facts]\n" + state["facts"],
                "messages": ["llm skipped (no key)"]}
    llm = ChatOpenAI(model="gpt-4o", temperature=0, max_tokens=600)
    reply = llm.invoke([
        SystemMessage(content=(
            "You are a medicinal-chemistry assistant. Given measured ChEMBL/PubChem data "
            "for one compound, write a 4-6 sentence assessment of its primary pharmacology, "
            "potency, and cardiac (hERG) safety risk. Quote the actual numbers you were "
            "given and do not invent any data.")),
        HumanMessage(content=state["facts"]),
    ])
    print(reply.content)
    return {"summary": reply.content, "messages": ["llm summary done"]}


graph = StateGraph(AgentState)
graph.add_node("fetch_structure",   node_fetch_structure)
graph.add_node("fetch_bioactivity", node_fetch_bioactivity)
graph.add_node("compile_facts",     node_compile_facts)
graph.add_node("llm_summary",       node_llm_summary)
graph.set_entry_point("fetch_structure")
graph.add_edge("fetch_structure",   "fetch_bioactivity")
graph.add_edge("fetch_bioactivity", "compile_facts")
graph.add_edge("compile_facts",     "llm_summary")
graph.add_edge("llm_summary",       END)
app = graph.compile()
print("Graph compiled successfully!")

## Run the prototype

In [ ]:
for compound in ["terfenadine", "ibuprofen", "caffeine"]:
    print("\n" + "=" * 60)
    print("AGENT RUN:", compound.upper())
    print("=" * 60)
    try:
        result = app.invoke({"compound": compound, "messages": []})
    except Exception as e:
        print(f"Error running agent: {e}")
        continue
    print("\n--- LLM summary ---")
    print(result["summary"])
    print("\n--- state log ---")
    for m in result["messages"]:
        print("  •", m)

## Key takeaways
- LangGraph makes agent state explicit — easier to debug than chain-based agents.
- Each node is a pure function: `state → partial state update`.
- Resolve a name to a **ChEMBL molecule ID first**, then query `activity` by
  `molecule_chembl_id` + `target_chembl_id` and keep only rows with a
  `pchembl_value` — that is what yields real, comparable numbers instead of empty
  results.
- Real-time APIs (PubChem, ChEMBL) give the agent current data beyond the LLM's
  training cutoff; the LLM node then turns those numbers into an assessment.
- The `llm_summary` node degrades to raw facts when no API key is present, so the
  graph stays runnable in any environment.